# Prep the Intermediary Files

This will include subseting the dataframe, and making the sex_at_birth and ndd columns into binary.

Currently, the sex_at_birth is Male or Female, but it needs to be 1 or 0 (respectively).

Currently, the {ndd}_DATE is a date variable or NaN, but needs to be 1 or 0 (respectively).

In [1]:
# import libraries
import pandas as pd
import numpy as np
import os

## Get the Code Names

In [ ]:
codes = pd.read_csv('../../data/labels.csv')
codes = codes[['FinnGen_Phenocode','ICD10_Codes','Cohort','Type','UKB_Description_For_Plots']]
codes = codes[codes['Cohort']=='UKB']
codes

In [ ]:
codes_list = list(codes['FinnGen_Phenocode'])
codes_list[:2]

In [7]:
# configure
ndd_list = ['PD','AD','DEM','VAS']
finngen_codes_not_found = []
results_table = []
path = ''

In [32]:
### testing to make sure it's correct..

In [ ]:
ndd = 'PD'
finngen_code = 'J10_INFLUPNEU'
t = pd.read_csv(f'{path}{ndd}_JULY_23_2026_ready_cox.csv', low_memory=False)
t.head()

In [ ]:
t['APOE'].value_counts()

In [ ]:
t['PD_DATE'].isna().value_counts()

## Configure all the Intermediary Files for the Mediation Analysis

In [27]:
# configure
ndd_list = ['PD','AD','DEM','VAS']
finngen_codes_not_found = []
results_table = []
path = ''

In [ ]:
# prep all the intermediary files..

for ndd in ndd_list:
    df = pd.read_csv(
        f'{path}{ndd}_JULY_23_2026_ready_cox.csv',
        low_memory=False
    )

    # # making sure they are binary - .astype(int) will make them 0 or 1, with 0 being the variable you've designated.
    df['NDD_BINARY'] = df[f'{ndd}_DATE'].notna().astype(int)
    # df['SEX_BINARY'] = (df['sex_at_birth'] == 'Male').astype(int)

    # get the number of cases & controls
    total_cases = (df['NDD_BINARY'] == 1).sum() # cases = 1
    total_controls = (df['NDD_BINARY'] == 0).sum() # controls = 0

    # for every code, collect only columns of interest
    for code in codes_list:
        # adding the 'real' column name
        code = f'QC0_{code}'

        # subsetting the df
        try:
            df_subset = df[
                [
                    'ID',
                    'NDD_BINARY',
                    'SEX',
                    code,
                    'APOE',
                    'date_of_birth',
                    'sex_at_birth',
                    f'{ndd}_DATE',
                    'DATE_OF_DEATH',
                    'recruit_date',
                    'tenure',
                    'age_at_tenure',
                ]
            ].copy()

            # save the df
            df_subset.to_csv(f'{path}/PREPPED_FOR_MEDIATION/{ndd}_{code}_prepped_df_for_mediation.csv', index=False)

            # creating a results df for exploration
            count_cases = ((df_subset[code] == 1) & (df_subset['NDD_BINARY'] == 1)).sum()
            count_controls = ((df_subset[code] == 1) & (df_subset['NDD_BINARY'] == 0)).sum()

            results_table.append({
                'NDD': ndd,
                'Code': code,
                'Cases_with_Code': count_cases,
                'Controls_with_Code': count_controls,
                'Total_Cases': total_cases,
                'Total_Controls': total_controls,
                'Total_Participants': len(df)
            })

            continue

        # just in case you cannot find a code in the labels.csv
        except KeyError:
            finngen_codes_not_found.append(code)
            message = f'COULD NOT FIND COLUMN FOR: {ndd} {code}'
            print(message)
            with open(f'{path}/PREPPED_FOR_MEDIATION/finngen_codes_not_found.txt', 'a') as f:
                f.write(message + '\n')
            continue

results_df = pd.DataFrame(results_table)

results_df.to_csv(f'{path}/PREPPED_FOR_MEDIATION/cases_for_mediation_results.csv', index=False)

results_df

In [ ]:
# TESTING check the output file

path2 = 'data/PREPPED_FOR_MEDIATION/'

t = pd.read_csv(f'{path2}/VAS_QC0_M13_SLE_prepped_df_for_mediation.csv')
t.APOE.value_counts()

# Understand the results_df a bit more

By evaluating this particular dataframe, we can help identify and justify why observations happened the way they did in the mediation analysis.

Ex.

- Some codes did not run. **explanation** they did not have enough cases to run
- Some codes had a bigger TE. **explanation** they had more cases so it was easier to see the TE
- Some codes had a TE but it wasn't signfiicant but it was on the threshold. **explanation** they had less cases so maybe with more, you would get sig
- Some codes you expected to have an effect (ex. influenza vs influenzapneumonia) did not show the same result. **explanation** maybe influenza didn't have as many cases.
- Some codes across the ndds didn't look the same way, results-wise. **explanation** they didn't run in a ndd because not enough cases or the cases between ndd #1 and ndd #2 had vastly different counts (maybe you care to interpret the code that had more cases)

In [ ]:
! pwd

In [45]:
import pandas as pd
t = pd.read_csv(f'{path}PREPPED_FOR_MEDIATION/cases_for_mediation_results.csv')

In [ ]:
# filter for only codes with >10 cases..

print('original: ',results_df.NDD.value_counts())
t = results_df.copy()
# t = t[t['Cases_with_Code']>=10]
# print('with >= 10 cases minimum: ',t.NDD.value_counts())
# t = t[t['Cases_with_Code']>=20]
# print('with >= 20 cases minimum: ',t.NDD.value_counts())
# t = t[t['Cases_with_Code']>=30]
# print('with >= 30 cases minimum: ',t.NDD.value_counts())
# t = t[t['Cases_with_Code']>=40]
# print('with >= 40 cases minimum: ',t.NDD.value_counts())
# t = t[t['Cases_with_Code']>=50]
# print('with >= 50 cases minimum: ',t.NDD.value_counts())

In [ ]:
# see how many have large case counts...we can infer simply through Prevelance Bias that

# test = t[t['NDD']=='VAS'] # you can switch through this like 'AD' or 'DEM' or 'PD' or 'VAS'
t = t.sort_values(by = 'Cases_with_Code', ascending=False)
t[:20]

# set up the bash script

In [ ]:
codes_list = [
'AB1_ANOGENITAL_HERPES_SIMPLEX',
'AB1_HERPES_SIMPLEX',
'AB1_OTHER_VIRAL',
'AB1_VIRAL_HEPATITIS',
'AB1_VIRAL_NOS',
'AB1_VIRAL_SKIN_MUCOUS_MEMBRANE',
'AB1_ZOSTER',
'E4_THYROIDITSUBAC',
'G6_MENINGVIR',
'H7_HERPESKERATITIS',
'I9_MYOCARD',
'INFLUENZA',
'J10_INFLUPNEU',
'MENINGITIS',
'ALLERG_RHINITIS',
'ASTHMA_MODE',
'CHIRBIL_PRIM',
'D3_ITP',
'D3_OTHERAPLASTICANAEMIA',
'E4_DM1',
'E4_GRAVES_STRICT',
'G6_GUILBAR',
'G6_MYASTHENIA',
'JUVEN_ARTHR',
'K11_COELIAC',
'L12_ALOPECAREATA',
'L12_BULLOUS',
'L12_LICHENPLANUS',
'L12_PSORIASIS',
'M13_ANKYLOSPON_ICD10',
'M13_NECROVASC',
'M13_RHEUMA',
'M13_RHEUMATISM',
'M13_SJOGREN',
'M13_SLE']
len(codes_list)

In [30]:
for ndd in ['PD','DEM','AD','VAS']:
    filename = f"11_{ndd}_MEDIATION_LOOP.sh"
    with open(filename, "w") as f:
        f.write("#!/bin/bash\n\n")
        f.write("# make sure to start the r-mediation conda env first!!\n")
        f.write(f'echo "STARTING ALL {ndd} CODES at $(TZ=America/New_York date)"\n')
        f.write(f'curl -d "{ndd} mediation started at $(TZ=America/New_York date)" https://ntfy.sh/\n\n') # this was a ntfy.sh notification, must configure yourself
    
        for code in codes_list:
            f.write(f'echo "STARTING {code} at $(TZ=America/New_York date)"\n')
            f.write(f'Rscript 08_Q3_MEDIATION_ANALYSIS.r {ndd} QC0_{code} 1000 > LOGS/{ndd}_QC0_{code}.log 2>&1\n')
            f.write(f'echo "FINISHED {code} at $(TZ=America/New_York date)"\n')
            f.write(f'curl -d "{code} mediation finished at $(TZ=America/New_York date)" https://ntfy.sh/\n\n')
    
        f.write(f'echo "FINISHED ALL {ndd} CODES at $(TZ=America/New_York date)"\n')
        f.write(f'curl -d "{ndd} mediation completed at $(TZ=America/New_York date)" https://ntfy.sh/\n')